In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

# ---- Settings ---- #
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 256
epochs = 3
bottleneck_sizes = [2, 4, 8, 16, 32, 64, 128, 256]
#full_data = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())


In [14]:
import torch
from torch.utils.data import Dataset
from functions import *
import os

class NoisySubspaceDataset(Dataset):
    def __init__(self, root='./data', train=True, download=True,
                 transform=None, n_samples=1000, d=10, k=3, noise_std=0.1):
        self.root = root
        self.train = train
        self.transform = transform
        self.d = d

        # Define path
        split = 'train' if train else 'test'
        self.filepath = os.path.join(root, f'noisy_subspace_{split}.pt')

        if not os.path.exists(self.filepath) or download:
            os.makedirs(root, exist_ok=True)
            self.data = self._generate_data(n_samples, d, k, noise_std)
            torch.save(self.data, self.filepath)
        else:
            self.data = torch.load(self.filepath)

    def _generate_data(self, n_samples, d, k, noise_std):
        # Orthonormal basis for k-dim subspace in R^d
        random_matrix = np.random.randn(d, k)
        basis, _ = np.linalg.qr(random_matrix)

        coeffs = np.random.randn(n_samples, k)
        clean = coeffs @ basis.T
        noise = noise_std * np.random.randn(*clean.shape)
        data = clean + noise

        return torch.tensor(data, dtype=torch.float32)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        x = self.data[idx]

        # Optional transform
        if self.transform:
            x = self.transform(x)

        return x

dataset = NoisySubspaceDataset( root='./data',
    train=True,
    download=True,
    transform=transforms.Lambda(lambda x: x))

#bottleneck_experiment(dataset, [2,4],batch_size = 12, epochs = 3)


In [15]:
dataset

In [16]:
bottleneck_experiment(dataset, [2,4],batch_size = 12, epochs = 3)

Training AE with bottleneck size 2...


ValueError: too many values to unpack (expected 2)